# Test All Methods - Prediction Logic Verification

**Mục đích**: Kiểm tra logic dự đoán của tất cả 7 phương pháp với test queries

**Test flow:**
1. Load tất cả models và ground truth
2. Implement prediction functions cho từng method (copy từ file 4 & 5)
3. Chạy test với vài query cụ thể
4. So sánh kết quả với ground truth để verify
5. Phân tích tại sao có sự mất cân bằng trong metrics

In [3]:
# Import libraries
import pandas as pd
import numpy as np
import pickle
import json
import os
from pathlib import Path
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

In [4]:
# Setup paths
DATA_PATH = r"E:\DS300-UIT-RecommenderSystem/Finalproject/data/all_recipes_final.csv"
MODELS_PATH = r"E:\DS300-UIT-RecommenderSystem/Finalproject/notebooks/Saved_models"
RAREC_PATH = r"E:\DS300-UIT-RecommenderSystem\Finalproject\notebooks\RA_Rec"
GROUND_TRUTH_PATH = r"E:\DS300-UIT-RecommenderSystem/Finalproject/notebooks/recommend_and_evaluation/eval_ground_truth.jsonl"

# Test parameters
TOP_K = 10  # Test với top 10 để dễ xem

print(f"Data path: {DATA_PATH}")
print(f"Models path: {MODELS_PATH}")
print(f"RARec path: {RAREC_PATH}")
print(f"Ground truth path: {GROUND_TRUTH_PATH}")
print(f"Top K for testing: {TOP_K}")

Data path: E:\DS300-UIT-RecommenderSystem/Finalproject/data/all_recipes_final.csv
Models path: E:\DS300-UIT-RecommenderSystem/Finalproject/notebooks/Saved_models
RARec path: E:\DS300-UIT-RecommenderSystem\Finalproject\notebooks\RA_Rec
Ground truth path: E:\DS300-UIT-RecommenderSystem/Finalproject/notebooks/recommend_and_evaluation/eval_ground_truth.jsonl
Top K for testing: 10


## 1. Load Data and Ground Truth

In [5]:
# Load recipes data
df = pd.read_csv(DATA_PATH)
df['recipe_id'] = df.index
print(f"Loaded {len(df)} recipes")
print(f"Columns: {df.columns.tolist()}")

Loaded 10263 recipes
Columns: ['title', 'type_of_food', 'link', 'description', 'ingredients', 'ingredients_normalized', 'step', 'note', 'num_of_ingredients', 'cook_time', 'num_of_people', 'calories', 'source', 'recipe_id']


In [6]:
# Load ground truth
ground_truth = []
with open(GROUND_TRUTH_PATH, 'r', encoding='utf-8') as f:
    for line in f:
        ground_truth.append(json.loads(line.strip()))

# Create ground truth dict
ground_truth_dict = {}
for item in ground_truth:
    query_id = item['query_id']
    relevant_docs = {}
    for doc in item['top10']:
        relevant_docs[doc['doc_id']] = doc['rel']
    ground_truth_dict[query_id] = relevant_docs

print(f"Loaded ground truth for {len(ground_truth_dict)} queries")

# Get query IDs list
query_ids = [item['query_id'] for item in ground_truth]
print(f"First 5 query_ids: {query_ids[:5]}")

Loaded ground truth for 200 queries
First 5 query_ids: [6302, 3779, 768, 3399, 4561]


## 2. Load All Models (6 similarity-based + 1 RARec)

In [7]:
# Method 1: TFIDF
print("Loading TFIDF...")
tfidf_similarity = np.load(os.path.join(MODELS_PATH, "TFIDF", "tfidf_similarity.npy"))
print(f"  TFIDF similarity matrix: {tfidf_similarity.shape}")

Loading TFIDF...
  TFIDF similarity matrix: (10263, 10263)


In [8]:
# Method 2: Ingredient_TFIDF
print("Loading Ingredient_TFIDF...")
ingredient_tfidf_similarity = np.load(os.path.join(MODELS_PATH, "Ingredient_TFIDF", "ingredient_tfidf_similarity.npy"))
print(f"  Ingredient TFIDF similarity matrix: {ingredient_tfidf_similarity.shape}")

Loading Ingredient_TFIDF...
  Ingredient TFIDF similarity matrix: (10263, 10263)


In [9]:
# Method 3: Keyword
print("Loading Keyword...")
keyword_similarity = np.load(os.path.join(MODELS_PATH, "Keyword", "keyword_similarity.npy"))
print(f"  Keyword similarity matrix: {keyword_similarity.shape}")

Loading Keyword...
  Keyword similarity matrix: (10263, 10263)


In [10]:
# Method 4: Hybrid (TFIDF + Ingredient)
print("Loading Hybrid...")
hybrid_similarity = np.load(os.path.join(MODELS_PATH, "Hybrid", "hybrid_similarity.npy"))
print(f"  Hybrid similarity matrix: {hybrid_similarity.shape}")

Loading Hybrid...
  Hybrid similarity matrix: (10263, 10263)


In [11]:
# Method 5: SBERT_FAISS
print("Loading SBERT...")
sbert_dir = os.path.join(MODELS_PATH, "SBERT_FAISS")
sbert_embeddings = np.load(os.path.join(sbert_dir, "recipe_embeddings.npy"))
print(f"  SBERT embeddings: {sbert_embeddings.shape}")

print("Computing SBERT similarity matrix...")
sbert_similarity = cosine_similarity(sbert_embeddings, sbert_embeddings)
print(f"  SBERT similarity matrix: {sbert_similarity.shape}")

Loading SBERT...
  SBERT embeddings: (10263, 768)
Computing SBERT similarity matrix...
  SBERT similarity matrix: (10263, 10263)


In [12]:
# Method 6: Hybrid_TFIDF_SBERT
print("Loading Hybrid_TFIDF_SBERT...")
hybrid_dir = os.path.join(MODELS_PATH, "Hybrid_TFIDF_SBERT")
with open(os.path.join(hybrid_dir, "config.json"), 'r', encoding='utf-8') as f:
    hybrid_config = json.load(f)

alpha = hybrid_config['alpha']
print(f"  Alpha: {alpha}")

# Combine TFIDF and SBERT
hybrid_tfidf_sbert_similarity = alpha * tfidf_similarity + (1 - alpha) * sbert_similarity
print(f"  Hybrid TFIDF+SBERT similarity matrix: {hybrid_tfidf_sbert_similarity.shape}")

Loading Hybrid_TFIDF_SBERT...
  Alpha: 0.5
  Hybrid TFIDF+SBERT similarity matrix: (10263, 10263)


In [13]:
# Method 7: RARec (Late Fusion)
print("Loading RARec components...")
model = SentenceTransformer('keepitreal/vietnamese-sbert')
print(f"  SBERT model loaded: {model.get_sentence_embedding_dimension()} dims")

embeddings_path = os.path.join(RAREC_PATH, "recipes_embeddings_list.pkl")
with open(embeddings_path, 'rb') as f:
    recipes_embeddings_list = pickle.load(f)
print(f"  Recipe embeddings list: {len(recipes_embeddings_list)} recipes")
print(f"  Example: Recipe 0 has {len(recipes_embeddings_list[0])} sentence embeddings")

Loading RARec components...
  SBERT model loaded: 768 dims
  Recipe embeddings list: 10263 recipes
  Example: Recipe 0 has 5 sentence embeddings


## 3. Implement Prediction Functions

In [14]:
def retrieve_top_k(query_idx, similarity_matrix, top_k=10, exclude_self=True):
    """
    Retrieve top K most similar items for a given query
    (Copy từ file 4_Evaluation.ipynb)
    """
    # Get similarity scores for query
    scores = similarity_matrix[query_idx].copy()
    
    # Exclude self if requested
    if exclude_self:
        scores[query_idx] = -1  # Set to -1 to exclude from results
    
    # Get top K indices (sorted by score descending)
    top_indices = np.argsort(scores)[::-1][:top_k]
    
    # Get doc_ids and scores
    results = []
    for idx in top_indices:
        doc_id = df.iloc[idx]['recipe_id']
        score = float(scores[idx])
        results.append({"doc_id": int(doc_id), "score": score})
    
    return results

In [15]:
def late_fusion_search_top_k(query_idx, model, recipes_embeddings_list, df, top_k=10, exclude_self=True):
    """
    Late Fusion search với top-K retrieval
    (Copy từ file 5_Evaluate_RARec.ipynb)
    """
    # 1. Get query recipe text
    query_recipe = df.iloc[query_idx]
    query_text = f"{query_recipe['title']}. {query_recipe['description']}"
    
    # 2. Encode query
    query_embedding = model.encode([query_text])
    query_embedding = query_embedding / np.linalg.norm(query_embedding)  # Normalize
    
    # 3. Calculate average similarity for EACH recipe
    recipe_scores = []
    
    for recipe_idx, dish_embeds in enumerate(recipes_embeddings_list):
        # Skip if no embeddings
        if len(dish_embeds) == 0:
            continue
        
        # Skip self if requested
        if exclude_self and recipe_idx == query_idx:
            continue
        
        # Normalize dish embeddings
        dish_embeds_norm = dish_embeds / np.linalg.norm(dish_embeds, axis=1, keepdims=True)
        
        # Compute cosine similarity với TẤT CẢ câu
        similarities = np.dot(dish_embeds_norm, query_embedding.T).flatten()
        
        # LATE FUSION: Average similarity
        avg_similarity = np.mean(similarities)
        
        recipe_scores.append({
            'doc_id': int(df.iloc[recipe_idx]['recipe_id']),
            'score': float(avg_similarity)
        })
    
    # 4. Sort by score descending and take top K
    recipe_scores.sort(key=lambda x: x['score'], reverse=True)
    
    return recipe_scores[:top_k]

## 4. Test with Sample Queries

In [16]:
# Select test queries
test_query_ids = query_ids[:3]  # Test với 3 query đầu tiên
print(f"Testing with {len(test_query_ids)} queries: {test_query_ids}")

Testing with 3 queries: [6302, 3779, 768]


In [17]:
# Create recipe_id to index mapping
recipe_id_to_idx = {recipe_id: idx for idx, recipe_id in enumerate(df['recipe_id'])}

In [18]:
# Define all methods
methods = {
    "TFIDF": tfidf_similarity,
    "Ingredient_TFIDF": ingredient_tfidf_similarity,
    "Keyword": keyword_similarity,
    "Hybrid": hybrid_similarity,
    "SBERT_FAISS": sbert_similarity,
    "Hybrid_TFIDF_SBERT": hybrid_tfidf_sbert_similarity
}

### 4.1. Test Query 1

In [19]:
# Select first test query
test_query_id = test_query_ids[0]
test_query_idx = recipe_id_to_idx[test_query_id]

print("="*100)
print(f"TEST QUERY ID: {test_query_id} (index: {test_query_idx})")
print("="*100)

# Show query info
query_recipe = df.iloc[test_query_idx]
print(f"\nQuery Recipe:")
print(f"  Title: {query_recipe['title']}")
print(f"  Description: {query_recipe['description'][:200]}...")
print(f"  Ingredients: {query_recipe['ingredients'][:150]}...")

# Show ground truth
if test_query_id in ground_truth_dict:
    gt = ground_truth_dict[test_query_id]
    print(f"\nGround Truth (Top 10 relevant docs):")
    for doc_id, rel in sorted(gt.items(), key=lambda x: x[1], reverse=True):
        doc_title = df[df['recipe_id'] == doc_id].iloc[0]['title']
        print(f"  Doc {doc_id} (rel={rel}): {doc_title}")
else:
    print("\nNo ground truth found for this query!")

TEST QUERY ID: 6302 (index: 6302)

Query Recipe:
  Title: Cá rô kho gừng thơm ngon đậm đà hương vị cho bữa cơm
  Description: Vào những ngày mưa gió, được quây quần cùng gia đình thưởng các món kho cùng nhau thì còn gì bằng. Điện máy XANH sẽ gợi ý đến bạn món cá rô kho gừng cực hấp dẫn, đậm vị và bắt cơm nhé, vào bếp thôi nà...
  Ingredients: ['600 gr Cá rô', '2 củ Gừng', '1 muỗng canh Dầu ăn', '2 trái Ớt hiểm', '2 muỗng canh Nước mắm', '1 muỗng canh Nước màu điều', '1 ít Gia vị thông dụng ...

Ground Truth (Top 10 relevant docs):
  Doc 6305 (rel=3): Cá thu kho gừng cay cay ngon miệng đậm đà dễ làm
  Doc 6264 (rel=3): Cá rô kho tộ thơm ngon đậm đà hấp dẫn cực đưa cơm
  Doc 6530 (rel=3): Món cá trê kho tiêu thơm ngon khó cưỡng cực đưa cơm
  Doc 6086 (rel=2): Thịt kho nghệ thơm ngon đậm đà dễ làm cho bữa cơm
  Doc 6319 (rel=2): Cá ngát kho gừng thơm ngon, đậm đà cho bữa cơm thêm tròn vị
  Doc 6290 (rel=2): Món cá diếc kho tiêu thơm ngon đậm đà hấp dẫn tại nhà
  Doc 6446 (rel=2): Thịt kho

In [20]:
# Test all 6 similarity-based methods
print("\n" + "="*100)
print("PREDICTIONS FROM ALL METHODS:")
print("="*100)

all_results = {}

for method_name, similarity_matrix in methods.items():
    print(f"\n{method_name}:")
    print("-"*80)
    
    results = retrieve_top_k(test_query_idx, similarity_matrix, top_k=TOP_K)
    all_results[method_name] = results
    
    # Count hits in ground truth
    if test_query_id in ground_truth_dict:
        gt_docs = set(ground_truth_dict[test_query_id].keys())
        hits = 0
        
        for rank, item in enumerate(results, 1):
            doc_id = item['doc_id']
            score = item['score']
            doc_title = df[df['recipe_id'] == doc_id].iloc[0]['title']
            
            is_relevant = "✓" if doc_id in gt_docs else " "
            if doc_id in gt_docs:
                hits += 1
                rel_score = ground_truth_dict[test_query_id][doc_id]
                print(f"  [{is_relevant}] {rank:2d}. Doc {doc_id:5d} (score={score:.4f}, rel={rel_score}) - {doc_title}")
            else:
                print(f"  [{is_relevant}] {rank:2d}. Doc {doc_id:5d} (score={score:.4f}, rel=0) - {doc_title}")
        
        print(f"\n  Hits in GT: {hits}/{TOP_K} (Precision@{TOP_K} = {hits/TOP_K:.2f})")


PREDICTIONS FROM ALL METHODS:

TFIDF:
--------------------------------------------------------------------------------
  [ ]  1. Doc  6547 (score=0.7862, rel=0) - Cá rô kho tương hột đậm đà thơm ngon dễ làm tại nhà
  [ ]  2. Doc  6605 (score=0.7485, rel=0) - Cá rô kho nghệ thơm lừng mềm ngon đậm đà hương vị
  [ ]  3. Doc  6160 (score=0.7423, rel=0) - Cá đù kho ngon miệng đậm đà hương vị cho bữa cơm cả nhà
  [ ]  4. Doc  7121 (score=0.6670, rel=0) - Cá rô rang muối dân dã lạ vị đổi món cho bữa cơm gia đình
  [ ]  5. Doc  4790 (score=0.6643, rel=0) - Cá rô nướng thơm ngon hấp dẫn đơn giản dễ làm cho bữa cơm
  [✓]  6. Doc  6264 (score=0.6634, rel=3) - Cá rô kho tộ thơm ngon đậm đà hấp dẫn cực đưa cơm
  [✓]  7. Doc  6305 (score=0.6629, rel=3) - Cá thu kho gừng cay cay ngon miệng đậm đà dễ làm
  [✓]  8. Doc  6319 (score=0.6604, rel=2) - Cá ngát kho gừng thơm ngon, đậm đà cho bữa cơm thêm tròn vị
  [✓]  9. Doc  6290 (score=0.6573, rel=2) - Món cá diếc kho tiêu thơm ngon đậm đà hấp dẫn tại n

In [21]:
# Test RARec method
print(f"\nRARec_Late_Fusion:")
print("-"*80)

rarec_results = late_fusion_search_top_k(test_query_idx, model, recipes_embeddings_list, df, top_k=TOP_K)
all_results["RARec_Late_Fusion"] = rarec_results

if test_query_id in ground_truth_dict:
    gt_docs = set(ground_truth_dict[test_query_id].keys())
    hits = 0
    
    for rank, item in enumerate(rarec_results, 1):
        doc_id = item['doc_id']
        score = item['score']
        doc_title = df[df['recipe_id'] == doc_id].iloc[0]['title']
        
        is_relevant = "✓" if doc_id in gt_docs else " "
        if doc_id in gt_docs:
            hits += 1
            rel_score = ground_truth_dict[test_query_id][doc_id]
            print(f"  [{is_relevant}] {rank:2d}. Doc {doc_id:5d} (score={score:.4f}, rel={rel_score}) - {doc_title}")
        else:
            print(f"  [{is_relevant}] {rank:2d}. Doc {doc_id:5d} (score={score:.4f}, rel=0) - {doc_title}")
    
    print(f"\n  Hits in GT: {hits}/{TOP_K} (Precision@{TOP_K} = {hits/TOP_K:.2f})")


RARec_Late_Fusion:
--------------------------------------------------------------------------------
  [ ]  1. Doc  6395 (score=0.7110, rel=0) - Món cá chép kho dưa chua thơm mềm, không tanh, chuẩn vị miền Bắc
  [ ]  2. Doc  6317 (score=0.6997, rel=0) - Cá ngừ kho nước dừa thơm ngon, đậm vị, bắt cơm
  [ ]  3. Doc  6123 (score=0.6980, rel=0) - Cá ngừ kho tỏi ớt đậm đà, bắt vị cho bữa cơm
  [ ]  4. Doc   298 (score=0.6884, rel=0) - Cá quả kho tương
  [ ]  5. Doc  6381 (score=0.6847, rel=0) - Cá linh kho nghệ ấm bụng, có màu vàng đẹp cho mâm cơm dân dã
  [ ]  6. Doc  6453 (score=0.6842, rel=0) - Cá kho riềng đậm đà, chắc thịt chuẩn vị miền Bắc với nồi gang
  [ ]  7. Doc  6546 (score=0.6826, rel=0) - Cá rô phi kho tương riềng đậm đà thơm nức mũi
  [ ]  8. Doc  6087 (score=0.6740, rel=0) - Cá nục kho kiểu Huế đậm vị, cay nồng cực hao cơm với nồi inox
  [ ]  9. Doc  6254 (score=0.6739, rel=0) - Cá diếc kho tương bần dân dã đậm đà hao cơm
  [ ] 10. Doc  6248 (score=0.6703, rel=0) - Cá chẽm kh

### 4.2. Comparison Summary for Query 1

In [22]:
# Summary comparison
print("\n" + "="*100)
print(f"SUMMARY COMPARISON FOR QUERY {test_query_id}:")
print("="*100)

if test_query_id in ground_truth_dict:
    gt_docs = set(ground_truth_dict[test_query_id].keys())
    
    summary_data = []
    for method_name, results in all_results.items():
        hits = sum(1 for item in results if item['doc_id'] in gt_docs)
        precision = hits / TOP_K
        
        # Calculate MRR
        mrr = 0.0
        for rank, item in enumerate(results, 1):
            if item['doc_id'] in gt_docs:
                mrr = 1.0 / rank
                break
        
        # Calculate nDCG
        dcg = 0.0
        for rank, item in enumerate(results, 1):
            doc_id = item['doc_id']
            rel = ground_truth_dict[test_query_id].get(doc_id, 0)
            dcg += rel / np.log2(rank + 1)
        
        ideal_rels = sorted(ground_truth_dict[test_query_id].values(), reverse=True)[:TOP_K]
        idcg = sum(rel / np.log2(rank + 1) for rank, rel in enumerate(ideal_rels, 1))
        ndcg = dcg / idcg if idcg > 0 else 0.0
        
        summary_data.append({
            'Method': method_name,
            'Hits': hits,
            f'Precision@{TOP_K}': f"{precision:.3f}",
            'MRR': f"{mrr:.3f}",
            f'nDCG@{TOP_K}': f"{ndcg:.3f}"
        })
    
    summary_df = pd.DataFrame(summary_data)
    print(summary_df.to_string(index=False))


SUMMARY COMPARISON FOR QUERY 6302:
            Method  Hits Precision@10   MRR nDCG@10
             TFIDF     4        0.400 0.167   0.294
  Ingredient_TFIDF     2        0.200 1.000   0.436
           Keyword     3        0.300 1.000   0.380
            Hybrid     4        0.400 1.000   0.529
       SBERT_FAISS     3        0.300 1.000   0.432
Hybrid_TFIDF_SBERT     3        0.300 0.250   0.244
 RARec_Late_Fusion     0        0.000 0.000   0.000


### 4.3. Test Multiple Queries

In [23]:
# Test với nhiều queries để xem pattern
print("\n" + "="*100)
print(f"TESTING WITH MULTIPLE QUERIES (first {len(test_query_ids)} queries)")
print("="*100)

multi_query_results = []

for test_qid in test_query_ids:
    test_qidx = recipe_id_to_idx[test_qid]
    
    print(f"\nQuery {test_qid}: {df.iloc[test_qidx]['title'][:50]}...")
    
    if test_qid not in ground_truth_dict:
        print("  No ground truth available")
        continue
    
    gt_docs = set(ground_truth_dict[test_qid].keys())
    
    query_results = {'query_id': test_qid}
    
    # Test all similarity-based methods
    for method_name, similarity_matrix in methods.items():
        results = retrieve_top_k(test_qidx, similarity_matrix, top_k=TOP_K)
        hits = sum(1 for item in results if item['doc_id'] in gt_docs)
        query_results[method_name] = hits
    
    # Test RARec
    rarec_results = late_fusion_search_top_k(test_qidx, model, recipes_embeddings_list, df, top_k=TOP_K)
    hits = sum(1 for item in rarec_results if item['doc_id'] in gt_docs)
    query_results["RARec_Late_Fusion"] = hits
    
    multi_query_results.append(query_results)
    
    # Print results
    for method_name in list(methods.keys()) + ["RARec_Late_Fusion"]:
        print(f"  {method_name:25s}: {query_results[method_name]}/{TOP_K} hits")

# Summary table
print("\n" + "="*100)
print("MULTI-QUERY SUMMARY:")
print("="*100)
multi_df = pd.DataFrame(multi_query_results)
print(multi_df.to_string(index=False))


TESTING WITH MULTIPLE QUERIES (first 3 queries)

Query 6302: Cá rô kho gừng thơm ngon đậm đà hương vị cho bữa c...
  TFIDF                    : 4/10 hits
  Ingredient_TFIDF         : 2/10 hits
  Keyword                  : 3/10 hits
  Hybrid                   : 4/10 hits
  SBERT_FAISS              : 3/10 hits
  Hybrid_TFIDF_SBERT       : 3/10 hits
  RARec_Late_Fusion        : 0/10 hits

Query 3779: Bánh táo yến mạch đơn giản thơm mềm chiêu đãi cả n...
  TFIDF                    : 0/10 hits
  Ingredient_TFIDF         : 1/10 hits
  Keyword                  : 1/10 hits
  Hybrid                   : 1/10 hits
  SBERT_FAISS              : 0/10 hits
  Hybrid_TFIDF_SBERT       : 0/10 hits
  RARec_Late_Fusion        : 0/10 hits

Query 768: Bỏ túi cách làm món khoai chiên mayo lắc mè cực gi...
  TFIDF                    : 0/10 hits
  Ingredient_TFIDF         : 0/10 hits
  Keyword                  : 0/10 hits
  Hybrid                   : 0/10 hits
  SBERT_FAISS              : 0/10 hits
  Hybrid_T